# Konfigurierte Variantenvergleiche publizieren

Wärme-, elektrische Endenergie-, Primärenergie- und Lebenszyklus-THG-Bilanz der drei Sanierungsvarianten. Die dynamischen Flussprofile stammen separat aus **Aspern F7**. Installation: `python -m pip install -e ".[publication]"`; statische Exporte benötigen Chrome.

Die Auswahl liegt in `website/_examples.yml`. Exakte Szenarioidentitäten bleiben erhalten; es werden keine Ergebnisdaten in der Konfiguration gespeichert.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "website/_examples.yml").exists()
sys.path.insert(0, str(ROOT / "scripts"))

In [ ]:
from website_examples import configuration, load_examples, annual_reports, balance_checks, input_differences
config = configuration()
project = load_examples(config)
reports = annual_reports(project)
input_differences(project)

## Systemgrenzen und Prüfung

Die Wärmebilanz und die lokale elektrische Nutzungsbilanz müssen schließen. Einspeisung und externe EV-Ladung gehören nicht zur lokalen Bedarfsdeckung. Die Primärenergiedifferenz ist dagegen ein Bewertungsergebnis. Die THG-Komponenten einschließlich negativer Beiträge müssen mit dem exportierten Gesamtwert übereinstimmen. In diesen Exporten sind nicht-elektrische Energieträger und PV-LCA null; abweichende künftige Konfigurationen benötigen eine geprüfte Darstellungsgrenze.

In [ ]:
checks = balance_checks(reports)
assert checks.loc[checks.Diagramm != "primary_energy_balance", "Differenz"].abs().max() < 1e-6
checks

In [ ]:
from pexl.plot import plotly
for report in reports.values():
    plotly.render(report).show()

## HTML, SVG und PNG

Der Website-Adapter materialisiert dieselben Reports und verwendet denselben Export. HTML-Dateien enthalten Plotly lokal. Statische Grafiken enthalten sichtbare Quellen- und Versionsangaben.

In [ ]:
from website_examples import build, provenance
from pexl.plot.export import export_figure
output = ROOT / "notebooks/local_exports/real_publication"
build(output, formats=("html",))
figure = plotly.render(reports["heat_balance"])
for fmt in ("svg", "png"):
    export_figure(figure, output / f"heat_balance.{fmt}", provenance=provenance(
        config["examples"], "Jahreswärmebilanz, kWh/(m² NGF a); getrennte Bedarfs- und Deckungssäulen."))
print(output)

## Pflege der vorhandenen Berichtsregistry

Serien werden in `data/schemas/1.13.1_dev/CHART_metadata.csv` gepflegt und mit dem bestehenden Generator erzeugt. Die folgende Zelle prüft die Reproduzierbarkeit, ohne das Paket zu überschreiben. Die THG-Gesamtbilanz nutzt bereits exportierte Größen, keine neue Bewertungsformel.

In [ ]:
from pexl.schema.report_codegen import generate_report_module_from_csv_dir
generated = generate_report_module_from_csv_dir(ROOT / "data/schemas/1.13.1_dev", output / "reports_check.py", version="1_13_1_dev", unknown_var_policy="raise")
assert generated.read_text(encoding="utf-8") == (ROOT / "src/pexl/schema/generated/reports_1_13_1_dev.py").read_text(encoding="utf-8")